# OneVoice V2 — English construction ASR
This is a future V2.1 gate and requires `MyDrive/onevoice_audio_v2_1/manifest.jsonl` with real speaker IDs and at least six voices. It does not use the Vietnamese V1 folders.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
WORK_ROOT = MYDRIVE / 'OneVoice'
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.environ['MODELSCOPE_CACHE'] = str(WORK_ROOT / 'model_cache/modelscope')
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'PyYAML', 'soundfile', 'librosa', 'funasr_onnx', 'modelscope'], check=True)
MANIFEST = MYDRIVE / 'onevoice_audio_v2_1/manifest.jsonl'
if not MANIFEST.is_file(): raise FileNotFoundError('English V2.1 audio has not been generated yet; run this notebook only after its manifest exists')
REPORT_ROOT = WORK_ROOT / 'reports/en_asr'
print('Source:', REPO, '| Data:', MANIFEST, '| Reports:', REPORT_ROOT)


In [ ]:
subprocess.run([sys.executable, 'scripts/audit_audio_dataset.py', str(MANIFEST), '--language', 'en', '--min-speakers', '6', '--require-realized-snr', '--expected-clean', '8064', '--expected-noisy', '16128', '--report-dir', str(REPORT_ROOT / 'audit')], check=True)
for audio in ('clean', 'noisy'):
    subprocess.run([sys.executable, 'scripts/benchmark_asr_v2.py', str(MANIFEST), '--direction', 'en2vi', '--split', 'test', '--audio', audio, '--denoiser', 'passthrough', '--report-dir', str(REPORT_ROOT / audio)], check=True)


In [ ]:
import json
{audio: json.loads((REPORT_ROOT / audio / 'aggregate.json').read_text(encoding='utf-8')) for audio in ('clean','noisy')}
